# Quickstart: Train, Save, and Reload

This notebook shows a tiny MLM smoke run using our UTF-8 tokenizer, how checkpoints/final outputs are organized under `runs/<RUN_ID>/...`, and how to reload the model/tokenizer.

In [ ]:
from utf8_tokenizer import create_tokenizer

In [ ]:
max_len = 64  # small for smoke/reuse demo

# Build tokenizer with optional seed texts to ensure demo characters exist
config = {
    "tokenizer_dir": "./tokenizers",
    "training": {"max_seq_length": max_len},
    "seed_texts": [
        "ا ل ح م د ل ل ه",
        "م ر ح ب ا",
        "ك ي ف ح ا ل ك",
        "ش ك ر ا",
        "أ ه ل ا و س ه ل ا",
        "ب س م ا ل ل ه",
        "ا ل ر ح م ن ا ل ر ح ي م"
    ],
}

tokenizer = create_tokenizer(config)

tokenizer.vocab_size

In [ ]:
# Version-agnostic TrainingArguments setup
from transformers import TrainingArguments
from inspect import signature
from pathlib import Path
import torch

# CKPT_DIR was defined in the run directories cell

ta_kwargs = dict(
    output_dir=str(CKPT_DIR),
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=50,
    report_to=[],
    fp16=torch.cuda.is_available(),
)
TA_params = signature(TrainingArguments.__init__).parameters
if "evaluation_strategy" in TA_params:
    ta_kwargs.update(dict(evaluation_strategy="epoch", save_strategy="epoch"))

args = TrainingArguments(**ta_kwargs)
args

In [ ]:
# Run directories layout
import datetime
from pathlib import Path

RUN_ID  = datetime.datetime.utcnow().strftime("%Y-%m-%d_%H-%M-%S")
RUN_DIR = Path("runs") / RUN_ID
CKPT_DIR = RUN_DIR / "checkpoints"
FINAL_DIR = RUN_DIR / "final"
ENV_DIR = RUN_DIR / "env"

for p in [CKPT_DIR, FINAL_DIR, ENV_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_DIR, CKPT_DIR, FINAL_DIR, ENV_DIR

In [ ]:
# Save final model + tokenizer
from transformers import Trainer

# Assume `trainer` exists after a brief train run in earlier cells
FINAL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_DIR))
# Save tokenizer in a self-contained way
try:
    tokenizer.save_pretrained(str(FINAL_DIR))
except Exception:
    # Fallback: at least save the vocab.json
    import json, os
    with open(os.path.join(FINAL_DIR, 'vocab.json'), 'w', encoding='utf-8') as f:
        json.dump(tokenizer.vocab, f, ensure_ascii=False, indent=2)
str(FINAL_DIR)

In [ ]:
# Reload model and tokenizer from FINAL_DIR
from transformers import BertForMaskedLM
from utf8_tokenizer import UTF8CharTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BertForMaskedLM.from_pretrained(str(FINAL_DIR)).to(device)
reload_tokenizer = UTF8CharTokenizer.from_pretrained(str(FINAL_DIR))

# Small inference demo
sample = "م ر ح ب ا"
enc = reload_tokenizer.encode(sample, padding=True, max_length=16)
import torch
with torch.no_grad():
    inputs = {
        "input_ids": torch.tensor([enc["input_ids"]]).to(device),
        "attention_mask": torch.tensor([enc["attention_mask"]]).to(device),
    }
    out = model(**inputs)
    loss = out.loss if hasattr(out, "loss") else None
loss

In [ ]:
# Record environment under ENV_DIR
import sys, subprocess, json

# Write python version
(ENV_DIR / "python_version.txt").write_text(sys.version, encoding="utf-8")

# Try to freeze requirements
try:
    req = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    (ENV_DIR / "requirements.lock").write_text(req, encoding="utf-8")
except Exception as e:
    (ENV_DIR / "requirements.lock").write_text(f"freeze_failed: {e}", encoding="utf-8")

# Minimal env summary
summary = {
    "python": sys.version.split(" ")[0],
    "cuda_available": bool(torch.cuda.is_available()),
}
(ENV_DIR / "env_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

str(ENV_DIR)

In [ ]:
# TinyMLMDataset and one-epoch smoke Trainer
from transformers import BertConfig, BertForMaskedLM, Trainer
import torch

# Tiny in-memory dataset using tokenizer.encode output
class TinyMLMDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, texts, max_length=64):
        self.samples = []
        for t in texts:
            enc = tokenizer.encode(t, padding=True, truncation=True, max_length=max_length)
            self.samples.append(enc)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

# Build tiny data
demo_texts = [
    "ا ل ح م د ل ل ه",
    "م ر ح ب ا",
    "ك ي ف ح ا ل ك",
    "ش ك ر ا",
    "أ ه ل ا و س ه ل ا",
]
train_ds = TinyMLMDataset(tokenizer, demo_texts, max_length=max_len)
eval_ds = TinyMLMDataset(tokenizer, demo_texts[:2], max_length=max_len)

# Minimal model
cfg = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=256,
    max_position_embeddings=max_len,
    pad_token_id=tokenizer.vocab.get(tokenizer.pad_token, 0),
)
model = BertForMaskedLM(config=cfg)

# Simple MLM data collator
class Collator:
    def __init__(self, tokenizer, p=0.15):
        self.tok = tokenizer
        self.p = p
        self.pad = tokenizer.vocab[tokenizer.pad_token]
        self.cls = tokenizer.vocab[tokenizer.cls_token]
        self.sep = tokenizer.vocab[tokenizer.sep_token]
        self.mask = tokenizer.vocab[tokenizer.mask_token]
    def __call__(self, batch):
        input_ids = torch.tensor([b['input_ids'] for b in batch])
        attention_mask = torch.tensor([b['attention_mask'] for b in batch])
        labels = input_ids.clone()
        prob = torch.full(labels.shape, self.p)
        special = (input_ids == self.pad) | (input_ids == self.cls) | (input_ids == self.sep)
        prob.masked_fill_(special, 0.0)
        mask_idx = torch.bernoulli(prob).bool()
        labels[~mask_idx] = -100
        rep = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & mask_idx
        input_ids[rep] = self.mask
        rnd = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & mask_idx & ~rep
        input_ids[rnd] = torch.randint(tokenizer.vocab_size, labels.shape, dtype=torch.long)[rnd]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

collator = Collator(tokenizer)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collator)
trainer.train(); print("SMOKE OK")